[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/05_machine_learning_fundamentals/07_data_leakage.ipynb)

# Bài 07 — Data Leakage: lỗi nguy hiểm nhất trong ML thực tế

**🎯 Mục tiêu**
- Định nghĩa data leakage và vì sao nó nguy hiểm: điểm offline tuyệt vời, triển khai thì thất bại.
- Tái hiện bằng code **6 dạng leakage** phổ biến và đo mức độ "ảo" của từng dạng.
- Checklist phòng tránh và phát hiện leakage.

> **Data leakage** xảy ra khi trong quá trình train/đánh giá, model được tiếp cận thông tin **không có sẵn tại thời điểm dự đoán thật**. Kết quả: điểm validation/test lạc quan giả tạo — và bạn chỉ phát hiện ra khi model đã được triển khai.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/05_machine_learning_fundamentals

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## Dạng 1 — Target leakage: feature chứa thông tin "tương lai"

Feature được tạo ra **sau** (hoặc **do**) sự kiện cần dự đoán. Ví dụ dự đoán churn với cột `called_cancellation_hotline` — khách hàng gọi tổng đài huỷ dịch vụ **chính vì** họ đã quyết định rời đi.

In [ ]:
from ml_course.datasets import make_customer_churn

churn = make_customer_churn(n=4000, seed=1)
churn["called_cancellation_hotline"] = ((churn["churn"] == 1) & (rng.random(len(churn)) < 0.8)).astype(int)

base_features = ["tenure_months", "monthly_charges", "support_calls", "n_products"]
for name, feats in [("không có feature rò rỉ", base_features), ("có called_cancellation_hotline", base_features + ["called_cancellation_hotline"])]:
    model = make_pipeline(StandardScaler(), LogisticRegression())
    auc = cross_val_score(model, churn[feats], churn["churn"], cv=5, scoring="roc_auc").mean()
    print(f"{name:35s} ROC-AUC = {auc:.3f}")

AUC tăng vọt — nhưng khi triển khai để **dự đoán trước** khách hàng nào sẽ rời đi, feature này **chưa tồn tại** (khách chưa gọi). Model vô dụng.

**Cách phát hiện:** với mỗi feature, hỏi *"giá trị này được ghi nhận **khi nào**? Tại thời điểm dự đoán nó đã có chưa?"*. Nghi ngờ mọi feature có tương quan "quá đẹp" với target.

## Dạng 2 — Chọn feature trên toàn bộ dữ liệu trước khi cross-validate

Ví dụ kinh điển (Hastie et al., *Elements of Statistical Learning* §7.10.2): dữ liệu **hoàn toàn ngẫu nhiên** — 50 mẫu, 5 000 feature nhiễu, nhãn ngẫu nhiên. Accuracy thật của mọi model = 50%.

In [ ]:
n, d = 50, 5000
X_noise = rng.normal(size=(n, d))
y_noise = rng.integers(0, 2, n)

# ❌ SAI: chọn 20 feature "tốt nhất" trên TOÀN BỘ dữ liệu, rồi mới cross-validate
X_selected = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
wrong = cross_val_score(LogisticRegression(), X_selected, y_noise, cv=5).mean()

# ✅ ĐÚNG: chọn feature BÊN TRONG mỗi fold (đặt trong Pipeline)
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())
right = cross_val_score(pipe, X_noise, y_noise, cv=5).mean()
print(f"Chọn feature trước CV: accuracy = {wrong:.2f}  ← dữ liệu NGẪU NHIÊN mà như model xuất sắc!")
print(f"Chọn feature trong CV: accuracy = {right:.2f}  ← đúng sự thật (~0.5)")

Khi chọn feature trên toàn bộ dữ liệu, ta đã chọn đúng những feature **tình cờ** tương quan với nhãn của **cả** các mẫu validation → validation không còn "chưa thấy". Mọi bước học từ dữ liệu (scaling, imputation, encoding, feature selection, PCA, oversampling, tuning) phải nằm **trong** vòng cross-validation → dùng **Pipeline** (chương 04).

## Dạng 3 — Oversampling / augmentation trước khi chia

Nhân bản (oversample) lớp hiếm **trước** khi chia train/test → các bản sao giống hệt nhau nằm ở **cả** train và test → model chỉ cần "nhớ" là đúng.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

from ml_course.datasets import make_fraud_transactions

fraud = make_fraud_transactions(n=8000, fraud_rate=0.03, seed=2)
Xf = fraud[["amount_vnd", "hour", "distance_from_home_km", "n_tx_last_24h", "card_age_days"]].to_numpy(float)
yf = fraud["is_fraud"].to_numpy()


def oversample(X, y, seed=0):
    r = np.random.default_rng(seed)
    pos = np.flatnonzero(y == 1)
    extra = r.choice(pos, (y == 0).sum() - len(pos), replace=True)
    idx = np.r_[np.arange(len(y)), extra]
    return X[idx], y[idx]


# ❌ oversample rồi mới chia
Xo, yo = oversample(Xf, yf)
Xtr, Xte, ytr, yte = train_test_split(Xo, yo, test_size=0.3, random_state=0)
tree = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
auc_wrong = roc_auc_score(yte, tree.predict_proba(Xte)[:, 1])

# ✅ chia trước, chỉ oversample tập train
Xtr, Xte, ytr, yte = train_test_split(Xf, yf, test_size=0.3, random_state=0, stratify=yf)
Xtr_o, ytr_o = oversample(Xtr, ytr)
tree = DecisionTreeClassifier(random_state=0).fit(Xtr_o, ytr_o)
auc_right = roc_auc_score(yte, tree.predict_proba(Xte)[:, 1])
print(f"Oversample trước khi chia: ROC-AUC = {auc_wrong:.3f}")
print(f"Chia trước, oversample train: ROC-AUC = {auc_right:.3f}")

## Dạng 4 — Leakage theo thời gian

Dữ liệu chuỗi thời gian: chia **ngẫu nhiên** cho model "nhìn thấy tương lai" — mỗi ngày trong tập test bị kẹp giữa các ngày train ở **cả hai phía**, model chỉ cần **nội suy** từ hàng xóm. Khi triển khai, ta luôn phải **ngoại suy** sang tương lai. Đánh giá đúng: **train quá khứ → test tương lai**.

Mô phỏng doanh số có **mức nền trôi dạt** theo thời gian (rất phổ biến: thị hiếu, đối thủ, kinh tế thay đổi):

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error

from ml_course.datasets import make_sales_series

sales = make_sales_series(n_days=3 * 365, seed=4)
level = np.exp(np.cumsum(rng.normal(0, 0.02, len(sales))))     # mức nền trôi dạt (random walk)
y_sales = sales["sales"] * level
X_time = pd.DataFrame({"t": np.arange(len(sales)), "dow": sales["date"].dt.dayofweek,
                       "doy": sales["date"].dt.dayofyear, "promo": sales["promo"]})

rf = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)
Xtr, Xte, ytr, yte = train_test_split(X_time, y_sales, test_size=0.2, random_state=0)           # ❌ ngẫu nhiên
mape_random = mean_absolute_percentage_error(yte, rf.fit(Xtr, ytr).predict(Xte))
cut = int(len(X_time) * 0.8)                                                                      # ✅ theo thời gian
pred_future = rf.fit(X_time.iloc[:cut], y_sales.iloc[:cut]).predict(X_time.iloc[cut:])
mape_time = mean_absolute_percentage_error(y_sales.iloc[cut:], pred_future)
print(f"Chia ngẫu nhiên    : MAPE = {mape_random:.1%}  ← ảo: model nội suy từ các ngày lân cận (kể cả tương lai)")
print(f"Chia theo thời gian: MAPE = {mape_time:.1%}  ← thực tế khi dự báo tương lai")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(sales["date"], y_sales, lw=0.7, label="thực tế")
ax.plot(sales["date"].iloc[cut:], pred_future, lw=1, label="dự báo (train trên quá khứ)")
ax.axvline(sales["date"].iloc[cut], color="k", ls="--")
ax.legend()
ax.set_title("Model cây không ngoại suy được mức nền mới")
plt.show()

Chia theo thời gian lộ ra hai vấn đề thật mà chia ngẫu nhiên che giấu hoàn toàn: model **dựa vào "ngày lân cận"** để đoán mức nền, và Random Forest **không ngoại suy** được (dự báo tương lai bị "kẹt" ở mức cuối cùng đã thấy). Chương 25 trình bày cách xử lý: feature trễ (lag), dự báo trên sai phân/tỉ lệ, backtesting.

## Dạng 5 — Leakage theo nhóm

Nhiều mẫu thuộc cùng một **đối tượng** (nhiều ảnh X-quang của cùng bệnh nhân, nhiều giao dịch của cùng khách hàng, nhiều câu của cùng người nói). Chia ngẫu nhiên → cùng một đối tượng ở cả train và test → model nhận ra **đối tượng** thay vì học quy luật tổng quát.

In [ ]:
n_patients, per_patient = 60, 20
patient_id = np.repeat(np.arange(n_patients), per_patient)
patient_style = rng.normal(size=(n_patients, 10))[patient_id]          # "dấu vân tay" riêng của mỗi bệnh nhân
label = rng.integers(0, 2, n_patients)[patient_id]                       # nhãn theo bệnh nhân, KHÔNG liên quan đặc trưng
Xg = patient_style + rng.normal(0, 0.3, size=patient_style.shape)

from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(1)
acc_random = cross_val_score(knn, Xg, label, cv=KFold(5, shuffle=True, random_state=0)).mean()
acc_group = cross_val_score(knn, Xg, label, cv=GroupKFold(5), groups=patient_id).mean()
print(f"KFold ngẫu nhiên: accuracy = {acc_random:.2f}  ← model nhận ra BỆNH NHÂN, không phải bệnh")
print(f"GroupKFold       : accuracy = {acc_group:.2f}  ← đúng sự thật (đặc trưng không liên quan tới bệnh)")

## Dạng 6 — Dữ liệu trùng lặp giữa train và test

Bản ghi trùng (hoặc gần trùng — cùng sản phẩm đăng hai lần, ảnh đã resize) ở cả hai tập → đo "khả năng nhớ" thay vì khả năng tổng quát hoá. Luôn **khử trùng lặp trước khi chia** (chương 03, bài 04).

## Các dạng khác cần nhớ

- **Target encoding** tính trên chính dữ liệu train (chương 04, bài 03).
- **ID / số thứ tự** tương quan với nhãn (dữ liệu được sắp theo nhãn rồi đánh ID).
- **Tune hyperparameter trên test** (bài 03) — leakage "trong quy trình", không phải trong feature.
- **Chuẩn hoá theo thống kê toàn bộ dữ liệu** — thường ảnh hưởng nhỏ, nhưng là thói quen xấu dẫn tới các dạng nghiêm trọng hơn.

## ✅ Checklist chống leakage

**Trước khi train**
- [ ] Với mỗi feature: *"có sẵn tại thời điểm dự đoán không?"* — ghi rõ thời điểm ghi nhận.
- [ ] Khử trùng lặp trước khi chia.
- [ ] Chia theo **thời gian** nếu dữ liệu có thời gian; theo **nhóm** nếu có nhiều mẫu/đối tượng.
- [ ] Chia train/test **trước** mọi bước học từ dữ liệu.

**Khi train**
- [ ] Mọi tiền xử lý / chọn feature / oversampling nằm **trong Pipeline**, trong vòng CV.
- [ ] Không bao giờ dùng test để chọn bất cứ thứ gì.

**Dấu hiệu cảnh báo**
- [ ] Kết quả "quá tốt so với kỳ vọng" (AUC ≈ 0.99 cho bài toán khó).
- [ ] Một feature đơn lẻ giải thích gần hết target.
- [ ] Điểm offline cao nhưng điểm khi triển khai thấp hơn nhiều.

## 🧠 Kiểm tra nhanh

1. Vì sao `SelectKBest` trước `cross_val_score` cho kết quả lạc quan, kể cả trên dữ liệu ngẫu nhiên?
2. Dự đoán bệnh nhân có tái nhập viện không — feature `discharge_summary_length` (độ dài báo cáo xuất viện) có an toàn không? Còn `num_followup_visits_next_month`?
3. Bạn có 10 ảnh/bệnh nhân. Nên dùng chiến lược chia dữ liệu nào?

## 📌 Tóm tắt

- Leakage = dùng thông tin không có tại thời điểm dự đoán → điểm số ảo.
- 6 dạng chính: target leakage, tiền xử lý/chọn feature ngoài CV, oversampling trước khi chia, thời gian, nhóm, trùng lặp.
- Phòng tránh: chia trước, Pipeline, chia theo thời gian/nhóm, kiểm tra thời điểm của từng feature, nghi ngờ kết quả quá tốt.

## 📚 Tài liệu tham khảo

- Kaufman, Rosset, Perlich — *Leakage in Data Mining* (KDD 2011)
- Hastie, Tibshirani, Friedman — *Elements of Statistical Learning*, §7.10.2 "The Wrong and Right Way to Do Cross-validation"
- Kapoor & Narayanan — *Leakage and the Reproducibility Crisis in ML-based Science* (Patterns, 2023)